# 新入职的员工的背景资料分析

In [39]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
plt.rcParams['font.sans-serif'] = ['SimHei']
%matplotlib inline
import pyecharts.options as opts
from pyecharts.charts import Line, Pie, Bar
from pyecharts.commons.utils import JsCode
from pyecharts.faker import Collector, Faker
from pyecharts.globals import ThemeType
from pyecharts.render import make_snapshot
from snapshot_selenium import snapshot

导入数据

In [2]:
this_year = pd.read_csv('Background2019.csv')
last_year = pd.read_csv('Background2018.csv')
this_year['label'] = 2019
last_year['label'] = 2018
this_year = this_year[['营业区','年龄','年龄区间','学历','性别','籍贯','婚姻状态','是否有小孩','在深居住时间','加盟新华前从事的职业', '同业从业时间', '同业职级', '加盟新华前年收入','加盟前是否购买过保险产品','label']]
this_year.columns = ['district', 'age', 'age_range','education','gender','hometown','marriage','kids','residency_time','occupation','same_job_duration','same_job_rank','income','insurrance','label']
this_year.sample

last_year = last_year[['营业区','年龄','年龄区间','学历','性别','籍贯','婚姻状态','是否有小孩','在深居住时间','加盟新华前从事的职业', '同业从业时间', '同业职级', '加盟新华前年收入','加盟前是否购买过保险产品','label']]
last_year.columns = ['district', 'age', 'age_range','education','gender','hometown','marriage','kids','residency_time','occupation','same_job_duration','same_job_rank','income','insurrance','label']
last_year.sample

all_data = [this_year, last_year]
all_data = pd.concat(all_data)

查看数据结构

In [3]:
all_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1676 entries, 0 to 485
Data columns (total 15 columns):
district             1676 non-null object
age                  1674 non-null float64
age_range            1674 non-null object
education            1674 non-null object
gender               1676 non-null object
hometown             1673 non-null object
marriage             1676 non-null object
kids                 1676 non-null object
residency_time       1675 non-null object
occupation           1676 non-null object
same_job_duration    611 non-null object
same_job_rank        610 non-null object
income               1676 non-null object
insurrance           1670 non-null object
label                1676 non-null int64
dtypes: float64(1), int64(1), object(13)
memory usage: 209.5+ KB


In [4]:
all_data.head()

,district,age,age_range,education,gender,hometown,marriage,kids,residency_time,occupation,same_job_duration,same_job_rank,income,insurrance,label
0,宝安,23.0,不足25岁,高中,女,云南,未婚,否,3-5年,行政办公人员,NaN,NaN,10万以内,否,2019
1,宝安,30.0,25-30岁,高中,男,广东,已婚,是,5年以上,个体经营者,NaN,NaN,10万以内,是,2019
2,宝安,48.0,45岁以上,高中,女,湖南,已婚,是,5年以上,个体经营者,NaN,NaN,10万以内,是,2019
3,龙岗,41.0,31-45岁,高中,男,江西,已婚,是,5年以上,个体经营者,NaN,NaN,10万以内,是,2019
4,龙华,37.0,31-45岁,高中,女,广西,已婚,是,1年内,同业销售,1-3年,营销员,10万以内,是,2019


处理异常项

In [21]:
all_data.groupby('occupation').occupation.count()
#发现异常，处理异常项
def get_string(string):
    if string == '个体经营者、其他行业销售':
        return '个体经营者'
    elif string == '同业销售、个体经营者':
        return '个体经营者'
    else:
        return string
all_data['occupation'] = all_data['occupation'].apply(get_string)

## 1.查看新入职员工的性别比例

### 1.1 总比例

In [50]:
# 2018-2019不同营业区男性和女性的比重的同比变化
this_year_ratio_female = all_data[(all_data['gender'] == '女') &(all_data['label'] == 2019)].gender.count()/all_data[all_data['label'] == 2019].gender.count()
last_year_ratio_female = all_data[(all_data['gender'] == '女') &(all_data['label'] == 2018)].gender.count()/all_data[all_data['label'] == 2018].gender.count()
this_year_ratio_female = np.round(this_year_ratio_female,2)
last_year_ratio_female = np.round(last_year_ratio_female,2)

this_year_ratio_male = 1 - this_year_ratio_female
last_year_ratio_male = 1 - last_year_ratio_female
# 同比去年女性新职工比重增加比例
print((this_year_ratio_female - last_year_ratio_female)/last_year_ratio_female)
# 同比去年男性职工比重增加比例
print((this_year_ratio_male - last_year_ratio_male)/last_year_ratio_male)


def gender_ratio() -> Pie:
    pie = (
        Pie()
        .add('', [('男',this_year_ratio_male),('女', this_year_ratio_female)])
        .set_global_opts(title_opts = opts.TitleOpts(title = '2019年新入职员工性别比例'))
        .set_series_opts(label_opts = opts.LabelOpts(formatter = '{b}: {c}'))
    )
    return pie
gender_ratio().render_notebook()


-0.0701754385964911
0.0930232558139533


In [6]:
def gender_ratio() -> Pie:
    pie = (
        Pie()
        .add('', [('男',np.round(last_year_ratio_male,2)),('女', last_year_ratio_female)])
        .set_global_opts(title_opts = opts.TitleOpts(title = '2018年新入职员工性别比例'))
        .set_series_opts(label_opts = opts.LabelOpts(formatter = '{b}: {c}'))
    )
    return pie
gender_ratio().render_notebook()

### 1.2 按区域查看比例

In [35]:
total_2019 = all_data[all_data['label']==2019].groupby('district').gender.count()
female_2019 = all_data[(all_data['label']==2019) & (all_data['gender']=='女')].groupby('district').gender.count()
male_2019 = total_2019 - female_2019
ratio_2019 = np.round(male_2019/female_2019,2)

total_2018 = all_data[all_data['label']==2018].groupby('district').gender.count()
female_2018 = all_data[(all_data['label']==2018) & (all_data['gender']=='女')].groupby('district').gender.count()
male_2018 = total_2018 - female_2018
ratio_2018 = np.round(male_2018/female_2018,2)

def gender_ratio() -> Bar:
    x = list(female_2019.index)
    bar = (
    Bar()
    .add_xaxis(x)
        # 这里变成list才正常
    .add_yaxis('2019年男女比例', list(ratio_2019.values))
    .add_yaxis('2018年男女比例', list(ratio_2018.values))
    .set_global_opts(title_opts = opts.TitleOpts(title = '2019年新入职员工性别比例(男:女)'),
                        yaxis_opts=opts.AxisOpts(name="比例"),
                        xaxis_opts=opts.AxisOpts(name="营业区"))
    )
        
    line = (
    Line()
    .add_xaxis(x)
    .add_yaxis('2019年男女比例', ratio_2019.values)
    .add_yaxis('2018年男女比例', ratio_2018.values)
    .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    )
        
    bar.overlap(line)
    return bar
gender_ratio().render_notebook()

## 2.查看新入职员工的年龄分布

### 2.1 数量

In [24]:
# 分年份
num2019 = list(all_data[(all_data['label'] == 2019)].groupby('occupation').occupation.count()[0:7])
print(num2019)

num2018 = list(all_data[(all_data['label'] == 2018)].groupby('occupation').occupation.count()[0:7])
print(num2018)


def new_staff_former_occupation() -> Bar:
    x = occupation
    bar = (
        Bar()
        .add_xaxis(x)
        .add_yaxis('2019', num2019)
        .add_yaxis('2018', num2018)
        .set_global_opts(title_opts=opts.TitleOpts(title="2018年和2019年新增员工之前从事的职业统计"))
    )
    line = (
        Line()
        .add_xaxis(x)
        .add_yaxis('2019', num2019)
        .add_yaxis('2018', num2018)
        .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    )
    bar.overlap(line)
    return bar
new_staff_former_occupation().render_notebook()


age2019 = list(all_data[all_data['label'] == 2019].groupby('age_range').age_range.count())
age2018 = list(all_data[all_data['label'] == 2018].groupby('age_range').age_range.count())
print('age2019:', age2019)
print('age2018:', age2018)
age2019_ratio = [np.round(age2019[i]/sum(age2019),2) for i in range(len(age2019))]
print(age2019_ratio)
age2018_ratio = [np.round(age2018[i]/sum(age2018),2) for i in range(len(age2018))]
print(age2018_ratio)

all_data[all_data['label'] == 2019].groupby('age_range').age_range.count()
age_range = ['25-30岁','31-45岁','45岁以上','不足25岁']
def new_staff_age_range() -> Bar:
    x = age_range
    bar = (
    Bar()
    .add_xaxis(x)
    .add_yaxis('2019', age2019)
    .add_yaxis('2018', age2018)
    .set_global_opts(title_opts = opts.TitleOpts(title = '2018年和2019年新增员工年龄数量'),
                        yaxis_opts=opts.AxisOpts(name="人数"),
                        xaxis_opts=opts.AxisOpts(name="年龄"))
    )
    
    line = (
    Line()
    .add_xaxis(x)
    .add_yaxis('2019', age2019)
    .add_yaxis('2018', age2018)
    .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    )
    bar.overlap(line)
    return bar
new_staff_age_range().render_notebook()


[296, 180, 281, 225, 30, 142, 36]
[120, 94, 157, 46, 22, 26, 21]
age2019: [275, 619, 175, 119]
age2018: [150, 265, 45, 26]
[0.23, 0.52, 0.15, 0.1]
[0.31, 0.55, 0.09, 0.05]


### 2.2 比例

In [36]:
age_range = ['25-30岁','31-45岁','45岁以上','不足25岁']
def new_staff_age_range() -> Bar:
    x = age_range
    bar = (
    Bar()
    .add_xaxis(x)
    .add_yaxis('2019', age2019_ratio)
    .add_yaxis('2018', age2018_ratio)
    .set_global_opts(title_opts = opts.TitleOpts(title = '2018年和2019年新增员工年龄比重'),
                        yaxis_opts=opts.AxisOpts(name="比例"),
                        xaxis_opts=opts.AxisOpts(name="年龄"))
    )
    
    line = (
    Line()
    .add_xaxis(x)
    .add_yaxis('2019', age2019_ratio)
    .add_yaxis('2018', age2018_ratio)
    .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    )
    bar.overlap(line)
    return bar
new_staff_age_range().render_notebook()

## 3. 查看新增员工的前职业分布

### 3.1 数量

In [61]:
occupation = list(all_data.groupby('occupation').occupation.count().index)
print(occupation)
def new_staff_former_occupation() -> Bar:
    bar = (
        Bar(init_opts = opts.InitOpts(width = "900px",height='500px'))#init_opts=opts.InitOpts(theme=ThemeType.LIGHT))
        .add_xaxis(occupation)
        .add_yaxis('2019年新入职员工前职业',num2019)
        .add_yaxis('2018年新入职员工前职业',num2018)
        .set_global_opts(title_opts = opts.TitleOpts(title = '新增员工的前职业数量'),
                        yaxis_opts=opts.AxisOpts(name="人数"),
                        xaxis_opts=opts.AxisOpts(name="职业"))
        )
        
        
    line = (
        Line()
        .add_xaxis(occupation)
        .add_yaxis('2019年新入职员工前职业', num2019)
        .add_yaxis('2018年新入职员工前职业', num2018)
        .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    )
    
    bar.overlap(line)
    return bar

new_staff_former_occupation().render_notebook()



['个体经营者', '其他', '其他行业销售', '同业销售', '社区女性', '行政办公人员', '财会人员']


### 3.2 比例

In [60]:
num2019_ratio = list(np.round(num2019[i]/sum(num2019),2) for i in range(len(num2019)))
num2018_ratio = list(np.round(num2018[i]/sum(num2018),2) for i in range(len(num2018)))

def new_staff_former_occupation_ratio() -> Bar:
    bar = (
        Bar(init_opts = opts.InitOpts(width = "900px",height='500px'))#init_opts=opts.InitOpts(theme=ThemeType.LIGHT))
        .add_xaxis(occupation)
        .add_yaxis('2019年新入职员工前职业比重',num2019_ratio)
        .add_yaxis('2018年新入职员工前职业比重',num2018_ratio)
        .set_global_opts(title_opts = opts.TitleOpts(title = '新增员工的前职业数量比重'),
                        yaxis_opts=opts.AxisOpts(name="比重"),
                        xaxis_opts=opts.AxisOpts(name="职业"))
        )
        
        
    line = (
        Line()
        .add_xaxis(occupation)
        .add_yaxis('', num2019_ratio)
        .add_yaxis('', num2018_ratio)
        .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    )
    
    bar.overlap(line)
    return bar

new_staff_former_occupation_ratio().render_notebook()

In [37]:
num2018_male = list(all_data[(all_data['label'] == 2018) & ( all_data['gender']== '男')].groupby('occupation').occupation.count())
print(num2018_male)
num2018_male.insert(4,0)
print('\t2018年男性各职业数量:',num2018_male)

num2018_female = list(all_data[(all_data['label'] == 2018) & ( all_data['gender']== '女')].groupby('occupation').occupation.count())
print('\t2018年女性各职业数量:',num2018_female)


num2018 = list(all_data[all_data['label']==2018].groupby('occupation').occupation.count())
print('\t2018年各职业数量:',num2018)

num2018_male_ratio = [num2018_male[i]/num2018[i] for i in range(len(num2018))]
num2018_male_ratio = list(np.round(num2018_male_ratio,2))
print('\t2018年男性新职员占比:',num2018_male_ratio)

num2018_female_ratio = [num2018_female[i]/num2018[i] for i in range(len(num2018))]
num2018_female_ratio = list(np.round(num2018_female_ratio,2))
print('\t2018年女性新职员占比:', num2018_female_ratio)


key = ['value','percent']
temp1 = []
for i in range(len(num2018)):
    temp1.append([num2018_male[i], num2018_male_ratio[i]])
temp2= []
for i in range(len(num2018)):
    temp2.append([num2018_female[i], num2018_female_ratio[i]])
    
# 用zip函数可以把[a,b],[c,d]转换成[(a,c),(b,d)]格式
# 用dict函数可以把[(a,c),(b,d)]转换成{'a':'c', 'b':'d'}格式
male_dict_list_2018 = []
for i in range(len(temp1)):
    male_dict_list_2018.append(dict(zip(key,temp1[i])))
print('男比例:',male_dict_list_2018)

female_dict_list_2018 = []
for i in range(len(temp2)):
    female_dict_list_2018.append(dict(zip(key,temp2[i])))
print('女比例:', female_dict_list_2018)

def new_staff_former_occupation_ratio() -> Bar:
    bar = (
        Bar(init_opts = opts.InitOpts(width = "900px",height='500px'))#init_opts=opts.InitOpts(theme=ThemeType.LIGHT))
        .add_xaxis(occupation)
        .add_yaxis('2019年新入职女员工比重',num2019_female_ratio)
        .add_yaxis('2018年新入职女员工比重',num2018_female_ratio)
        .set_global_opts(title_opts = opts.TitleOpts(title = '新增女员工的前职业比重'),
                        yaxis_opts=opts.AxisOpts(name="比例"),
                        xaxis_opts=opts.AxisOpts(name="职业"))
        )
        
        
    line = (
        Line()
        .add_xaxis(occupation)
        .add_yaxis('2019年新入职女员工比重', num2019_female_ratio)
        .add_yaxis('2018年新入职女员工比重', num2018_female_ratio)
        .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    )
    
    bar.overlap(line)
    return bar

new_staff_former_occupation_ratio().render_notebook()

[54, 51, 70, 27, 5, 3]
	2018年男性各职业数量: [54, 51, 70, 27, 0, 5, 3]
	2018年女性各职业数量: [66, 43, 87, 19, 22, 21, 18]
	2018年各职业数量: [120, 94, 157, 46, 22, 26, 21]
	2018年男性新职员占比: [0.45, 0.54, 0.45, 0.59, 0.0, 0.19, 0.14]
	2018年女性新职员占比: [0.55, 0.46, 0.55, 0.41, 1.0, 0.81, 0.86]
男比例: [{'value': 54, 'percent': 0.45}, {'value': 51, 'percent': 0.54}, {'value': 70, 'percent': 0.45}, {'value': 27, 'percent': 0.59}, {'value': 0, 'percent': 0.0}, {'value': 5, 'percent': 0.19}, {'value': 3, 'percent': 0.14}]
女比例: [{'value': 66, 'percent': 0.55}, {'value': 43, 'percent': 0.46}, {'value': 87, 'percent': 0.55}, {'value': 19, 'percent': 0.41}, {'value': 22, 'percent': 1.0}, {'value': 21, 'percent': 0.81}, {'value': 18, 'percent': 0.86}]
